# CKD Data Preprocessing

This notebook preprocesses the Chronic Kidney Disease (CKD) dataset before model training.

## Objectives

- Load the raw CKD dataset
- Split into train/test **before** imputing — imputation statistics (median/mode) must
  come from the training set only, or test-set information leaks into training features
- Handle missing values (train-fit imputers, applied to both splits)
- Encode categorical variables (train-fit encoders, applied to both splits)
- Save separate processed train/test CSVs for machine learning experiments

Output:

- 
- 

In [16]:
import os
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

## Load Raw Data

In [17]:
df = pd.read_csv("../../data/raw/chronic_kidney_disease.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (400, 25)


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,36.0,1.2,NaN,NaN,15.4,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,18.0,0.8,NaN,NaN,11.3,38.0,6000.0,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,53.0,1.8,NaN,NaN,9.6,31.0,7500.0,NaN,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,56.0,3.8,111.0,2.5,11.2,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,26.0,1.4,NaN,NaN,11.6,35.0,7300.0,4.6,no,no,no,good,no,no,ckd


In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     391 non-null    float64
 1   bp      388 non-null    float64
 2   sg      353 non-null    float64
 3   al      354 non-null    float64
 4   su      351 non-null    float64
 5   rbc     248 non-null    str    
 6   pc      335 non-null    str    
 7   pcc     396 non-null    str    
 8   ba      396 non-null    str    
 9   bgr     356 non-null    float64
 10  bu      381 non-null    float64
 11  sc      383 non-null    float64
 12  sod     313 non-null    float64
 13  pot     312 non-null    float64
 14  hemo    348 non-null    float64
 15  pcv     329 non-null    float64
 16  wbcc    294 non-null    float64
 17  rbcc    269 non-null    float64
 18  htn     398 non-null    str    
 19  dm      398 non-null    str    
 20  cad     398 non-null    str    
 21  appet   399 non-null    str    
 22  pe      399 n

In [19]:
print("Missing values before replacing '?'")
print(df.isnull().sum())

print("\nDataset shape:", df.shape)

Missing values before replacing '?'
age        9
bp        12
sg        47
al        46
su        49
rbc      152
pc        65
pcc        4
ba         4
bgr       44
bu        19
sc        17
sod       87
pot       88
hemo      52
pcv       71
wbcc     106
rbcc     131
htn        2
dm         2
cad        2
appet      1
pe         1
ane        1
class      0
dtype: int64

Dataset shape: (400, 25)


## Clean String Values

Replace the dataset's "?" missing-value marker with NaN, and strip stray whitespace —
this only touches formatting, not statistics, so it is safe to do before the split.

In [20]:
df.replace("?", np.nan, inplace=True)

df = df.apply(
    lambda col: col.str.strip()
    if col.dtype == object
    else col
)

print("Missing values after replacing '?'")
print(df.isnull().sum())

Missing values after replacing '?'
age        9
bp        12
sg        47
al        46
su        49
rbc      152
pc        65
pcc        4
ba         4
bgr       44
bu        19
sc        17
sod       87
pot       88
hemo      52
pcv       71
wbcc     106
rbcc     131
htn        2
dm         2
cad        2
appet      1
pe         1
ane        1
class      0
dtype: int64


## Train / Test Split (before imputation and encoding)

Splitting here — before any statistic is computed — is what actually prevents leakage.
Stratifying directly on the raw  labels ( / ) keeps the same 80/20
class balance in both splits.

In [21]:
numeric_columns = [
    "age", "bp", "sg", "al", "su", "bgr", "bu", "sc",
    "sod", "pot", "hemo", "pcv", "wbcc", "rbcc"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

categorical_columns = [
    col for col in df.columns
    if col not in numeric_columns and col != "class"
]

print("Numerical Columns:")
print(numeric_columns)
print()
print("Categorical Columns:")
print(categorical_columns)

Numerical Columns:
['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc']

Categorical Columns:
['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']


In [22]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["class"]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print()
print("Train class balance:")
print(train_df["class"].value_counts())
print()
print("Test class balance:")
print(test_df["class"].value_counts())

Train shape: (320, 25)
Test shape : (80, 25)

Train class balance:
class
ckd       200
notckd    120
Name: count, dtype: int64

Test class balance:
class
ckd       50
notckd    30
Name: count, dtype: int64


## Imputation (fit on train only)

Median for numeric columns, mode for categorical columns — both computed **only** from
, then applied to both  and . This is the fix: previously
these statistics were computed on the full 400-row dataset, which meant the test set
quietly influenced the values used to fill in training rows (and vice versa).

In [23]:
num_imputer = SimpleImputer(strategy="median")
num_imputer.fit(train_df[numeric_columns])

train_df[numeric_columns] = num_imputer.transform(train_df[numeric_columns])
test_df[numeric_columns] = num_imputer.transform(test_df[numeric_columns])

cat_imputer = SimpleImputer(strategy="most_frequent")
cat_imputer.fit(train_df[categorical_columns])

train_df[categorical_columns] = cat_imputer.transform(train_df[categorical_columns])
test_df[categorical_columns] = cat_imputer.transform(test_df[categorical_columns])

print("Missing values in train after imputation:", train_df.isnull().sum().sum())
print("Missing values in test after imputation :", test_df.isnull().sum().sum())

Missing values in train after imputation: 0
Missing values in test after imputation : 0


## Categorical Encoding (fit on train only)

Each categorical feature is label-encoded using classes seen in . Any category
in  that was never seen in training falls back to the train-set mode instead of
crashing — this is a defensive edge case, not expected to trigger on this dataset since
every categorical column here is low-cardinality (2 classes) and the split is stratified,
but it is the correct way to handle it if it ever did.

In [24]:
encoders = {}

for col in categorical_columns:
    le = LabelEncoder()
    le.fit(train_df[col].astype(str))
    encoders[col] = le

    train_df[col] = le.transform(train_df[col].astype(str))

    fallback = train_df[col].mode()[0]
    test_df[col] = test_df[col].astype(str).apply(
        lambda v: le.transform([v])[0] if v in le.classes_ else fallback
    )

# Explicit mapping for the target instead of LabelEncoder's alphabetical order.
# LabelEncoder would assign ckd=0, notckd=1 (alphabetical), which silently makes
# recall/precision/F1/AUC treat the HEALTHY class as positive. CKD = 1 always.
target_map = {"notckd": 0, "ckd": 1}

train_df["class"] = train_df["class"].astype(str).str.strip().map(target_map)
test_df["class"] = test_df["class"].astype(str).str.strip().map(target_map)

assert train_df["class"].isna().sum() == 0, "Unexpected class label in train set"
assert test_df["class"].isna().sum() == 0, "Unexpected class label in test set"

print("Categorical encoding completed.")
print("Target mapping -> notckd: 0, ckd: 1")

Categorical encoding completed.
Target mapping -> notckd: 0, ckd: 1


## Final Checks

In [25]:
print("Final train missing values:", train_df.isnull().sum().sum())
print("Final test missing values :", test_df.isnull().sum().sum())

assert train_df.isnull().sum().sum() == 0, "NaNs remain in train set"
assert test_df.isnull().sum().sum() == 0, "NaNs remain in test set"

train_df.head()

Final train missing values: 0
Final test missing values : 0


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,59.0,60.0,1.020,0.0,0.0,1,1,0,0,113.0,23.0,1.1,139.0,3.5,15.3,54.0,6500.0,4.9,0,0,0,0,0,0,0
1,76.0,70.0,1.015,3.0,4.0,1,0,1,0,120.0,164.0,9.7,131.0,4.4,10.2,30.0,11300.0,3.4,1,1,1,1,1,0,1
2,70.0,90.0,1.015,0.0,0.0,1,1,0,0,144.0,125.0,4.0,136.0,4.6,12.0,37.0,8200.0,4.5,1,1,0,1,1,0,1
3,28.0,60.0,1.025,0.0,0.0,1,1,0,0,79.0,50.0,0.5,145.0,5.0,17.6,51.0,6500.0,5.0,0,0,0,0,0,0,0
4,23.0,80.0,1.020,0.0,0.0,1,1,0,0,99.0,46.0,1.2,142.0,4.0,17.7,46.0,4300.0,5.5,0,0,0,0,0,0,0


## Save Processed Train/Test Sets

In [26]:
os.makedirs("../../data/processed", exist_ok=True)

train_df.to_csv("../../data/processed/chronic_kidney_disease_train.csv", index=False)
test_df.to_csv("../../data/processed/chronic_kidney_disease_test.csv", index=False)

print("Processed train/test sets saved successfully.")

Processed train/test sets saved successfully.


In [27]:
print("=" * 50)
print("Preprocessing completed successfully")
print("=" * 50)

print("\nTrain Shape:", train_df.shape)
print("Test Shape :", test_df.shape)

print("\nTrain Target Distribution")
print(train_df["class"].value_counts())

print("\nTest Target Distribution")
print(test_df["class"].value_counts())

Preprocessing completed successfully

Train Shape: (320, 25)
Test Shape : (80, 25)

Train Target Distribution
class
1    200
0    120
Name: count, dtype: int64

Test Target Distribution
class
1    50
0    30
Name: count, dtype: int64
